# NB03 · Semana 5 — Simulacro de incidente y auditoría (para E2)

**Ingeniería de Datos — LCD · UCA Rosario · 2026**

Dataset: **VATSIM** (`bronze/vatsim`). Usamos **time travel de Delta** + **DuckDB** para auditar un incidente de ingesta (una corrida duplicada) y derivar las reglas de validación de la E2.

> **Nota técnica.** La sintaxis `VERSION AS OF` / `delta_as_v2` del enunciado es orientativa (estilo Databricks). DuckDB no lee el historial Delta por SQL; lo hacemos con la librería `deltalake` (historial y time travel) y DuckDB para el álgebra relacional (`EXCEPT`, `GROUP BY`).

## 0. Configuración del acceso al lake

In [ ]:
# === Configuración del entorno — NO tocar ===
import os, sys
_d = os.getcwd()
while True:
    if os.path.exists(os.path.join(_d, "entorno.py")):
        sys.path.insert(0, _d); break
    _p = os.path.dirname(_d)
    if _p == _d:
        raise FileNotFoundError("No encuentro entorno.py subiendo desde el directorio actual.")
    _d = _p
from entorno import BUCKET, STORAGE_OPTIONS, info
info()

BRONZE = f's3://{BUCKET}/bronze/vatsim'              # tu tabla real (no se toca)
INCIDENTE = f's3://{BUCKET}/bronze/_incidente_vatsim'  # copia donde reproducimos el bug
CLAVE = ["cid", "snapshot_ts"]
print("Tabla real:", BRONZE)
print("Tabla del simulacro:", INCIDENTE)

## 1. Reproducir el incidente sobre una copia de tu lake

Tu ingesta es idempotente, así que `bronze/vatsim` **no** tiene duplicados. Para tener una anomalía que auditar, reproducimos el bug descrito por el profe ("el script corrió dos veces por error") sobre una **copia de tus propios datos**: tomamos los dos últimos snapshots reales de VATSIM y reingerimos el último **dos veces**, salteando el watermark.

Esto no modifica `bronze/vatsim`.

In [ ]:
import pandas as pd
from deltalake import DeltaTable, write_deltalake

real = DeltaTable(BRONZE, storage_options=STORAGE_OPTIONS).to_pandas()
ts = sorted(real["snapshot_ts"].dropna().unique())
assert len(ts) >= 2, "Necesitás al menos 2 snapshots en bronze/vatsim: corré 'ingerir' unas veces en el NB02."
ts_ant, ts_act = ts[-2], ts[-1]
snap_ant = real[real["snapshot_ts"] == ts_ant]
snap_act = real[real["snapshot_ts"] == ts_act]
print(f"snapshot anterior: {ts_ant}  ({len(snap_ant)} filas)")
print(f"snapshot actual  : {ts_act}  ({len(snap_act)} filas)")

# Reconstruimos la tabla del simulacro desde cero:
write_deltalake(INCIDENTE, snap_ant, mode="overwrite", storage_options=STORAGE_OPTIONS)  # corrida normal
write_deltalake(INCIDENTE, snap_act, mode="append",    storage_options=STORAGE_OPTIONS)  # corrida normal
# --- EL INCIDENTE: la misma corrida se ejecuta de nuevo (duplicado) ---
write_deltalake(INCIDENTE, snap_act, mode="append",    storage_options=STORAGE_OPTIONS)
print("Incidente reproducido en", INCIDENTE)

## Paso 1 — Detectar el salto (auditoría de versiones)

Leemos el historial de la tabla Delta: cuántas versiones hay, qué operación fue cada una, y cuántas filas sumó la última corrida respecto de la anterior.

In [ ]:
dt = DeltaTable(INCIDENTE, storage_options=STORAGE_OPTIONS)

hist = pd.DataFrame(dt.history())
cols = [c for c in ["version", "timestamp", "operation", "operationMetrics"] if c in hist.columns]
print("=== Historial de la tabla ===")
print(hist[cols].to_string(index=False))

ult = dt.version()
print(f"\n=== Filas por versión (última = v{ult}) ===")
for v in range(max(0, ult - 3), ult + 1):
    n = DeltaTable(INCIDENTE, version=v, storage_options=STORAGE_OPTIONS).to_pyarrow_dataset().count_rows()
    print(f"  v{v}: {n} filas")

n_act = DeltaTable(INCIDENTE, version=ult,     storage_options=STORAGE_OPTIONS).to_pyarrow_dataset().count_rows()
n_ant = DeltaTable(INCIDENTE, version=ult - 1, storage_options=STORAGE_OPTIONS).to_pyarrow_dataset().count_rows()
print(f"\nLa última corrida sumó {n_act - n_ant} filas (de v{ult-1} a v{ult}).")

## Paso 2 — Aislar la anomalía con un JOIN temporal (time travel)

Cargamos la versión **actual** y la **anterior** con time travel y las comparamos en DuckDB.

- `EXCEPT` por la clave de negocio → qué claves entraron que no estaban antes.
- `GROUP BY ... HAVING COUNT(*) > 1` → qué claves quedaron duplicadas.

> Comparamos por `cid, snapshot_ts` (la clave natural), **no** por `*`: las columnas de metadata (`_run_id`, `_ingested_at`) cambian en cada corrida y ensuciarían el diff.

In [ ]:
import duckdb

df_actual   = DeltaTable(INCIDENTE, version=ult,     storage_options=STORAGE_OPTIONS).to_pandas()
df_anterior = DeltaTable(INCIDENTE, version=ult - 1, storage_options=STORAGE_OPTIONS).to_pandas()

con = duckdb.connect()
con.register("v_actual", df_actual)
con.register("v_anterior", df_anterior)

print("=== ¿Qué claves entraron de más respecto de la versión anterior? ===")
entraron = con.sql('''
    SELECT cid, snapshot_ts, callsign FROM v_actual
    EXCEPT
    SELECT cid, snapshot_ts, callsign FROM v_anterior
''').df()
if len(entraron):
    print(entraron.to_string(index=False))
else:
    print("Ninguna clave nueva: todo lo que 'entró' ya existía → fue una REINGESTA (corrida duplicada).")

print("\n=== Duplicados por clave natural en la versión actual ===")
dups = con.sql('''
    SELECT cid, snapshot_ts, COUNT(*) AS n
    FROM v_actual
    GROUP BY cid, snapshot_ts
    HAVING COUNT(*) > 1
    ORDER BY n DESC, cid
''').df()
print(dups.to_string(index=False) if len(dups) else "Sin duplicados.")

total = len(df_actual)
unicas = con.sql("SELECT COUNT(*) FROM (SELECT DISTINCT cid, snapshot_ts FROM v_actual)").fetchone()[0]
print(f"\nDiagnóstico: {total} filas totales vs {unicas} claves únicas → {total - unicas} filas duplicadas.")

## Paso 3 — Diagnóstico para la E2: reglas de validación en dbt

Lo que el simulacro dejó en evidencia se traduce en dos reglas **obligatorias** para la capa silver (dbt), la semana que viene:

**Regla 1 — Unicidad de la clave natural.** No puede existir más de una fila con la misma combinación `(cid, snapshot_ts)`: un piloto aparece una sola vez por snapshot. Es la regla que el incidente violó.

**Regla 2 — Campos clave no nulos.** `cid` y `snapshot_ts` no pueden ser nulos: sin ellos, la clave no identifica al registro y la Regla 1 no se puede verificar.

Así se implementan en el `schema.yml` del modelo silver (dbt):

```yaml
version: 2
models:
  - name: stg_vatsim
    columns:
      - name: cid
        tests:
          - not_null            # Regla 2
      - name: snapshot_ts
        tests:
          - not_null            # Regla 2
    tests:
      - dbt_utils.unique_combination_of_columns:   # Regla 1
          combination_of_columns: [cid, snapshot_ts]
```

Si no querés depender de `dbt_utils`, la Regla 1 se puede escribir como test singular en `tests/assert_vatsim_sin_duplicados.sql` (dbt espera que no devuelva filas):

```sql
SELECT cid, snapshot_ts, COUNT(*) AS n
FROM {{ ref('stg_vatsim') }}
GROUP BY cid, snapshot_ts
HAVING COUNT(*) > 1
```

**Reglas candidatas extra** (para mencionar en la defensa): `latitude` entre -90 y 90, `longitude` entre -180 y 180, `altitude` numérica y no negativa.

## Cierre

- **Paso 1:** el historial Delta mostró una versión de más y un salto de filas sin claves nuevas.
- **Paso 2:** el time travel + `EXCEPT`/`GROUP BY` confirmó que fueron duplicados por reingesta, no datos nuevos.
- **Paso 3:** de ahí salen las dos reglas dbt (unicidad de `(cid, snapshot_ts)` y no-nulos).

> La tabla `bronze/vatsim` quedó intacta; todo el simulacro corrió sobre `bronze/_incidente_vatsim`.

In [ ]:
import duckdb
con = duckdb.connect('/home/jovyan/work/warehouse.duckdb')
tablas = con.execute("""
    select table_name, table_type
    from information_schema.tables
    where table_schema = 'main'
    order by table_name
""").fetchdf()
con.close()
print(tablas)